
<h1 style="line-height:0.5;">Classifying Athlete Performance from Historical Olympic Competitions</h1>

<p>CMSC320: Introduction to Data Science Spring 2026 Final Project</p>

<table style="margin-left:auto; margin-right:auto; text-align:center; table-layout:fixed; width:75%; border:none; border-collapse: collapse">
    <tr>
    <td style="border:none; padding:25px">
        <img 
            src="https://media.licdn.com/dms/image/v2/D5603AQFE6iUECGLtkQ/profile-displayphoto-shrink_800_800/profile-displayphoto-shrink_800_800/0/1718273641091?e=1779926400&v=beta&t=VxG4ThN4Hrh9bQKOCcp0x_4ztl_i0vpGZsmdMerkjAI"
            style="width:75%"
        >
    </td>
    <td style="border:none; padding:25px">
        <img 
            src="https://media.licdn.com/dms/image/v2/D4E03AQF2MWkM042FqA/profile-displayphoto-crop_800_800/B4EZw7YfbYHsAI-/0/1770522803316?e=1779926400&v=beta&t=3rA1rVPoIN1gtGme6IxoNmrhbP_sFwd8OLqhX2SYTUU"
            style="width:75%"
        >
    </td>
    <td style="border:none; padding:25px">
        <img 
            src="https://media.licdn.com/dms/image/v2/D4E03AQFwYbleqrkF6Q/profile-displayphoto-crop_800_800/B4EZx3jfT4JcAI-/0/1771532319029?e=1779926400&v=beta&t=LPuT0gg2OoZX4IRAmYTy9ngU4dK8KS8lLNaDz8u-TKI"
            style="width:75%"
        >
    </td>
    <td style="border:none; padding:25px">
        <img 
            src="https://media.licdn.com/dms/image/v2/D4D03AQEDfaNhA_h1pQ/profile-displayphoto-shrink_200_200/B4DZYQWSQoHwAc-/0/1744031000218?e=2147483647&v=beta&t=pM3sjvfHL5qh44rj29eH9RAyZXD7MoIgPS-SdXRKTWo"
            style="width:75%"
        >
    </td>
    <td style="border:none; padding:25px">
        <img 
            src="https://media.licdn.com/dms/image/v2/D4D03AQFM6ZMFgcMYpw/profile-displayphoto-shrink_800_800/profile-displayphoto-shrink_800_800/0/1715135307072?e=1779926400&v=beta&t=dDpfs5S7Xe7cnwdbgsMNSqJqb4txN1ouP-GMNsE8xQI"
            style="width:75%"
        >
    </td>
  </tr>
  <tr>
    <th style="border:none; padding:1px">Netra Thirumuruhan</th>
    <th style="border:none; padding:1px">Harrison Padgett</th>
    <th style="border:none; padding:1px">Jim Barry</th>
    <th style="border:none; padding:1px">Disha Ramesh</th>
    <th style="border:none; padding:1px">Josh Tagle</th>
  </tr>
  <tr>
    <td style="border:none; padding:1px">A, B</td>
    <td style="border:none; padding:1px">A, B, C</td>
    <td style="border:none; padding:1px">C, F</td>
    <td style="border:none; padding:1px">D, E</td>
    <td style="border:none; padding:1px">C, G</td>
  </tr>
  <tr>
    <td style="border:none; padding:1px">Summary goes here...</td>
    <td style="border:none; padding:1px">Summary goes here...</td>
    <td style="border:none; padding:1px">Summary goes here...</td>
    <td style="border:none; padding:1px">Summary goes here...</td>
    <td style="border:none; padding:1px">Summary goes here...</td>
  </tr>
</table>


<h2>2. Introduction</h2>
<p>
The Olympic Games bring together thousands of athletes from over 200 countries every four years. With over 126 years of recorded history, the dataset of Olympic results is large enough to ask data driven questions about what separates medalists from non-medalists. This project attempts to do that.
</p><p>
The central question is can we predict whether an Olympic athlete will win a medal based on their physical profile, demographic background, and event? We use a dataset of over 155,000 athlete biographies and event results spanning Athens 1896 through Tokyo 2020 to build and evaluate a classification model for this task.
</p><p>
Our analysis proceeds in three stages. First, we conduct exploratory data analysis and hypothesis testing to identify whether specific factors, host nation status, athlete sex, and event structure, have statistically measurable relationships with medal outcomes. Second, we engineer features from the raw data and build a classification pipeline using several standard models, evaluating performance under stratified cross-validation. Third, we interpret the results.
</p><p>
The short answer to our central question is that physical and demographic features are weak predictors of medal outcomes. No model tested achieved an F1 above roughly 0.39 on the medal class, despite including height, weight, age, country, sport, prior appearances, and event-level statistics. This is a meaningful result: it suggests that Olympic success is not well-explained by the attributes recorded here, and that factors outside this dataset, training quality, in-competition performance, coaching, likely account for most of the variance in who medals and who does not.
</p>

In [7]:
# INSTALL DEPENDENCIES

%pip install kagglehub -q

# IMPORT LIBRARIES

import kagglehub
import pandas as pd
import os

Note: you may need to restart the kernel to use updated packages.


<h2>3. Data Curation</h2>

<p>
    Our dataset is sourced from Kaggle:
    <a href="https://www.kaggle.com/datasets/muhammadehsan02/126-years-of-historical-olympic-dataset">
    126 Years of Historical Olympic Dataset</a>. It comprises six CSV files covering every Olympic
    Games from Athens 1896 through Tokyo 2020/Beijing 2022.
</p>
<p>
    The files we use are: <code>Olympic_Athlete_Biography.csv</code> (155,861 athletes with height,
    weight, country, and birth date), <code>Olympic_Athlete_Event_Details.csv</code> (per-athlete
    event entries with medal outcomes and team sport flags), <code>Olympic_Games_Summary.csv</code>
    (edition metadata including year), and <code>Olympic_Country_Profiles.csv</code> (NOC-to-country
    name mapping).
</p>
<p>
    Several preprocessing steps were required before analysis. First, we merged the biography,
    event, and games tables on <code>athlete_id</code> and <code>edition_id</code> to produce a
    single denormalized record per athlete-event appearance. Second, weight values were stored as
    both single numbers and ranges (e.g., <code>"69-77"</code>); we converted ranges to their
    midpoint mean and coerced all values to float. Third, birth dates were parsed to datetime with
    error coercion. Fourth, we dropped all records prior to 1960, since height and weight missingness
    exceeded 45% for those years, imputing that many values would bias the physical attribute
    features we rely on for ML. Finally, remaining missing heights and weights were imputed with
    the per-sex, per-sport mean, with a global mean fallback for any residual nulls. The result is
    <code>biography_pruned</code>: a clean DataFrame of athlete-event appearances from 1960-2022
    ready for EDA and modeling.
</p>

In [8]:
# INSTALL DEPENDENCIES

%pip install kagglehub -q

# IMPORT LIBRARIES

import kagglehub
import pandas as pd
import os

# LOADING IN THE DATASET

path = kagglehub.dataset_download("muhammadehsan02/126-years-of-historical-olympic-dataset")

path1 = os.path.join(path, "Olympic_Country_Profiles.csv")
path2 = os.path.join(path, "Olympic_Athlete_Biography.csv")
path3 = os.path.join(path, "Olympic_Athlete_Event_Details.csv")
path4 = os.path.join(path, "Olympic_Event_Results.csv")
path5 = os.path.join(path, "Olympic_Medal_Tally_History.csv")
path6 = os.path.join(path, "Olympic_Games_Summary.csv")

countries_df = pd.read_csv(path1)
biography_df = pd.read_csv(path2)
events_df = pd.read_csv(path3)
results_df = pd.read_csv(path4)
medal_tally_df = pd.read_csv(path5)
olympics_df = pd.read_csv(path6)

Note: you may need to restart the kernel to use updated packages.


In [9]:
# PRINT DESCRIPTIVE STATISTICS

def explore(df, name):
    print(f"\n{name.upper()}")
    print("\nSHAPE:", df.shape)
    print("\nDATASET INFORMATION:")
    print(df.info())
    print("\nDESCRIPTIVE STATISTICS:")
    print(df.describe())
    print("\nMISSING VALUES:")
    print(df.isnull().sum())
    #print("\nPREVIEW:")
    #display(df.head(5))

explore(countries_df, "Countries")
explore(biography_df, "Biographies")
explore(events_df, "Events")
explore(results_df, "Results")
explore(medal_tally_df, "Medal Tallies")
explore(olympics_df, "Olympic Games")




COUNTRIES

SHAPE: (235, 2)

DATASET INFORMATION:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   noc      235 non-null    object
 1   country  235 non-null    object
dtypes: object(2)
memory usage: 3.8+ KB
None

DESCRIPTIVE STATISTICS:
        noc      country
count   235          235
unique  234          235
top     ROC  Afghanistan
freq      2            1

MISSING VALUES:
noc        0
country    0
dtype: int64

BIOGRAPHIES

SHAPE: (155861, 10)

DATASET INFORMATION:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155861 entries, 0 to 155860
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   athlete_id     155861 non-null  int64  
 1   name           155861 non-null  object 
 2   sex            155861 non-null  object 
 3   born           151808 non-null  object 
 4   heigh

<h3> Events (events_df) and Olympic (olympics_df) data cleaning </h3>

<p><i>events_df</i> holds information about the edition of the Games, sport, event specifics and information on the athlete's performance, including their position, medal won, and whether the event was a team sport. </p>
<p><i>olympics_df</i>  holds information about the Olympic Games edition, year hosted, host city, country, URLS for edition and country flags, and dates for the start, end, and competition periods </p>


In [10]:
#fill in missing medal values values
events_df['medal'] = events_df['medal'].fillna('No Medal')

#delete games that didnt happen due to war
olympics_df = olympics_df[olympics_df['isHeld'] != 'Not held due to war']
olympics_df = olympics_df.drop('isHeld', axis=1)

#drop country_flag_url (kinda useless) and 'country_noc' when the country name is given
olympics_df = olympics_df.drop('country_flag_url', axis=1)
biography_df = biography_df.drop('country_noc', axis=1)
medal_tally_df = medal_tally_df.drop('country_noc', axis=1)

In [11]:
# Update 1900 summer olympics_df
olympics_df.loc[1, ['start_date', 'end_date']] = ['14 May', '28 October']

#update 1904 summer olympics_df
olympics_df.loc[2, ['start_date', 'end_date']] = ['1 July', '26 November']

#converting start and end_date to datetime
olympics_df['start_date'] = pd.to_datetime(olympics_df['start_date'] + ' ' + olympics_df['year'].astype(str),format='mixed',
    dayfirst=True,
    errors='coerce').dt.date
olympics_df['end_date'] = pd.to_datetime(olympics_df['end_date'] + ' ' + olympics_df['year'].astype(str), format='mixed',
    dayfirst=True,
    errors='coerce').dt.date

#delete games that don't exist yet
olympics_df.dropna(subset=['start_date', 'end_date'], inplace=True)

#olympics_df.head()

#adding in competing country
events_df = events_df.merge(
    countries_df[['noc', 'country']],
    left_on='country_noc',
    right_on='noc',
    how='left'
)

events_df = events_df.drop(['noc', 'country_noc'], axis=1)
events_df = events_df.rename(columns={'country': 'country_name'})
#events_df.head()


<p>This section cleans the Olympic and Events datasets so they are easier to analyze and merge. Missing values in the `medal` column are filled with "No Medal" because blank medal entries represent athletes who did not win a medal. This also makes the column easier to convert into a binary target variable later. Olympic Games that were not held due to war are removed because they do not contain real competition results. After that, unnecessary or duplicate columns, such as 'country_flag_url' and repeated 'country_noc' columns, are dropped to simplify the datasets. We also converted the start and end date columns into date time and deleted any future games that havent happened yet. Finally, the events dataset is merged with the countries dataset so each event record includes the full country name instead of only the NOC code.</p>

<h3> Biography (biography_df) Data Cleaning</h3>
<p><i>biography_df</i>  holds athlete information like name, sex, birth date, height, weight, and country</p>

In [12]:
#adding in competing country
events_df = events_df.merge(
    countries_df[['noc', 'country']],
    left_on='country_noc',
    right_on='noc',
    how='left'
)

events_df = events_df.drop(['noc', 'country_noc'], axis=1)
events_df = events_df.rename(columns={'country': 'country_name'})
events_df.head()

KeyError: 'country_noc'

In [ ]:
#merging in columns from events_df and olympics_df to biography_df
biography_df = (
    biography_df
    .merge(
        events_df[['athlete_id', 'edition_id', 'pos', 'medal', 'event', 'result_id', 'isTeamSport', 'edition']],
        on='athlete_id',
        how='left'
    )
    .merge(
        olympics_df[['edition_id', 'year']],
        on='edition_id',
        how='left'
    )
)

#merge in sport
biography_df = biography_df.merge(
    events_df[['athlete_id', 'sport']],
    on='athlete_id',
    how='left'
)

# clean birth dates into standard format
# some dates are messy/ not formatted correctly so we coerce
# note: shouldn't be that many with ill-formatted dates, so it's fine
biography_df['born'] = pd.to_datetime(biography_df['born'], errors='coerce')

#biography_df.head()

In [ ]:
#weights are not all numbers
non_numeric = biography_df[
    pd.to_numeric(biography_df['weight'], errors='coerce').isna()
]

#WHY ARE THEY RANGES WE HAVE TO CLEAN
non_numeric['weight'].unique()

In [ ]:
#making weight column into integers
w = biography_df['weight'].astype(str).str.replace('\xa0', '', regex=False).str.strip()
w = w.str.replace(r'^(\d+),(\d+)$', r'\1.\2', regex=True)

r = w.str.extract(r'^(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)$').astype(float).mean(axis=1)
c = w.str.extract(r'^(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)$').astype(float).mean(axis=1)
s = pd.to_numeric(w, errors='coerce')

biography_df['weight'] = s.fillna(r).fillna(c)
biography_df['weight'].unique()

In [ ]:
#making weight column into integers
w = biography_df['weight'].astype(str).str.replace('\xa0', '', regex=False).str.strip()
w = w.str.replace(r'^(\d+),(\d+)$', r'\1.\2', regex=True)

r = w.str.extract(r'^(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)$').astype(float).mean(axis=1)
c = w.str.extract(r'^(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)$').astype(float).mean(axis=1)
s = pd.to_numeric(w, errors='coerce')

biography_df['weight'] = s.fillna(r).fillna(c)
biography_df['weight'].unique()

In [ ]:
import matplotlib.pyplot as plt
missing_by_year = biography_df.groupby('year')[['height','weight']].apply(lambda x: x.isna().mean())

plt.figure(figsize=(7, 4))

plt.plot(
    missing_by_year.index,
    missing_by_year.values,
    marker="o"
)

plt.title("Average Missing Height/Weight Data by Year")
plt.xlabel("Year")
plt.ylabel("Average Proportion Missing")
plt.ylim(0, 1)
plt.grid(True)

plt.show()

<h5>Because records before 1960 contain a higher proportion of missing height and weight values, we removed those earlier years from the dataset. This helps create a cleaner dataset for later visualizations and machine learning analysis.</h5>

<p> We chose not to impute the missing height and weight values because the missingness is not random. Most missing values occur before 1960, which shows that older Olympic records were less complete. Imputing those with an average could introduce artificial data and bias the analysis, so we chose to prune the dataset to include records from 1960 onward instead of relying on large amounts of imputed values (since over half the data for height and weight were missing for those years).
</p>

In [ ]:
#only includes values after 1960
biography_pruned = biography_df[biography_df['year'] >= 1960]

#replacing NaN weights and heights with the mean based on the sex and sport
for col in ['height', 'weight']:
    biography_pruned[col] = biography_pruned.groupby(['sex', 'sport'])[col] \
        .transform(lambda x: x.fillna(x.mean()))

    biography_pruned[col] = biography_pruned[col].fillna(biography_pruned[col].mean())

#dropping duplicates
biography_pruned = biography_pruned.drop_duplicates()

#biography_pruned.head()


<h2>4. Exploratory Data Analysis</h2>

<p>
    Before building a predictive model, we need to understand the structure of the data. Our EDA
    pursues three questions: 
    (1) Do host nations win more medals than they do when competing abroad?
    (2) Has female athlete representation at the Olympics increased significantly over time?
    (3) In mixed-gender events, do men and women win medals at equal rates?
</p>


<h3>Group by summer/winter, gender, etc. </h3>

In [ ]:
# seperate by summer/winter olympics
df_summer = biography_pruned[biography_pruned['edition'].str.contains('Summer', case=True, na=False)]
df_winter = biography_pruned[biography_pruned['edition'].str.contains('Winter', case=True, na=False)]

# seperate by gender
gender = biography_pruned['sex'].value_counts()
df_male = biography_pruned[biography_pruned['sex'] == 'Male']
df_female = biography_pruned[biography_pruned['sex'] == 'Female']

#gender mean and median
gender_mean = biography_pruned.groupby('sex')[['height', 'weight']].mean()
gender_median = biography_pruned.groupby('sex')[['height', 'weight']].median()

#greatest gender diff in sports
sports_by_gender = biography_pruned.groupby(['sport', 'sex']).size().unstack(fill_value=0)
sports_by_gender['diff'] = (sports_by_gender['Male'] - sports_by_gender['Female']).abs()
df_sorted = sports_by_gender.sort_values(by='diff', ascending=True)

# seperate by gender and summer/winter olympics
df_summerMale = df_summer[df_summer['sex'] == "Female"]
df_summerMale = df_summer[df_summer['sex'] == "Male"]
df_winterFemale = df_winter[df_winter['sex'] == "Female"]
df_winterMale = df_winter[df_winter['sex'] == "Male"]

# country attendance
df_country = biography_pruned.groupby('country').size().sort_values(ascending=False)

# country by medal 
medals_only = biography_pruned[biography_pruned['medal'] != 'No Medal']
df_medal = medals_only.groupby('country')['medal'].count().sort_values(ascending=False)

print("difference in gender counts: \n" , gender, "\n")
print("difference in mean by height/weight: \n", gender_mean, "\n")
print("difference in median by height/weight: \n", gender_median, "\n")
print("sports with big gender diff: ", df_sorted.head(5), "\n")
print("highest attending countries: ", df_country, "\n")
print("highest medaling countries: ", df_medal, "\n")


<h3>Conclusion #1 - Home Field Advantage For Medals</h3>
<p>Testing to see whether or not a host country has an advantage for winning medals by comparing their home performance to their away performance. Medals include bronze, silver, and gold.</p>

In [ ]:
import pandas as pd
import scipy.stats as stats

# Grab the host's 3 letter noc code
host_info_raw = olympics_df[['edition_id', 'country_noc']]

# Use countries_df to translate noc back into the full string
host_info = host_info_raw.merge(countries_df, left_on='country_noc', right_on='noc', how='left')

# Keep edition id and country name
host_info = host_info[['edition_id', 'country']].rename(columns={'country': 'host_country'})

# Merge this into our medal tally dataset
merged_tally = medal_tally_df.merge(host_info, on='edition_id', how='left')

# Create a True/False column if the competing country matches the host country that year
merged_tally['is_host'] = merged_tally['country'] == merged_tally['host_country']

# Look only at countries that have hosted AT LEAST ONCE
host_nations_list = merged_tally[merged_tally['is_host'] == True]['country'].unique()
filtered_tally = merged_tally[merged_tally['country'].isin(host_nations_list)]

# Average number of medals won AWAY
avg_away_medals = filtered_tally[filtered_tally['is_host'] == False].groupby('country')['total'].mean().reset_index()
avg_away_medals.rename(columns={'total': 'avg_medals_away'}, inplace=True)

# Average number of medals won HOME
avg_home_medals = filtered_tally[filtered_tally['is_host'] == True].groupby('country')['total'].mean().reset_index()
avg_home_medals.rename(columns={'total': 'avg_medals_home'}, inplace=True)

# Merge averages
paired_df = avg_home_medals.merge(avg_away_medals, on='country')

# Run test
t_stat, p_val = stats.ttest_rel(paired_df['avg_medals_home'], paired_df['avg_medals_away'])

print(paired_df.head())
print(f"Number of countries tested: {len(paired_df)}")
print(f"Average Medals HOME: {paired_df['avg_medals_home'].mean():.2f}")
print(f"Average Medals AWAY: {paired_df['avg_medals_away'].mean():.2f}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.5f}")


<p>Graph the results using horizontal lines to better represent the home field advantage when it comes to medals</p>

In [ ]:
import matplotlib.pyplot as plt

# Sort by advantage in descending order
paired_df['advantage'] = paired_df['avg_medals_home'] - paired_df['avg_medals_away']
sorted_df = paired_df.sort_values('advantage', ascending=True).reset_index(drop=True)

# Set the y-axis label range
y_range = range(1, len(sorted_df) + 1)

plt.figure(figsize=(12, 10)) # Needs to be tall to comfortably fit all countries

# Draw horizontal lines for advantage
plt.hlines(y=y_range, 
           xmin=sorted_df['avg_medals_away'], 
           xmax=sorted_df['avg_medals_home'], 
           color='grey', alpha=0.5, linewidth=2)

# Plot away avg medals
plt.scatter(sorted_df['avg_medals_away'], y_range, 
            color='darkgrey', alpha=1, s=80, label='Away Average', zorder=3)

# Plot home avg medals
plt.scatter(sorted_df['avg_medals_home'], y_range, 
            color='gold', alpha=1, s=80, edgecolors='black', label='Home Average', zorder=3)


# Add y axis names, x axis labels, and title
plt.yticks(y_range, sorted_df['country'], fontsize=11)
plt.xlabel("Average Medals Won per Olympics", fontsize=13, fontweight='bold')
plt.title('The Olympic Home Field Advantage\n(Line represents difference in averages)', 
          fontsize=16, fontweight='bold')

# Background lines for easier viewing
plt.grid(axis='x', linestyle='--', alpha=0.5)

# Add legend
plt.legend(fontsize=12, loc='lower right')
plt.tight_layout()

plt.show()


Note that some countries have marginal advantages or away advantages (Switzerland and Yugoslavia, for example)

<h3> Conclusion #2 - The inclusion of Female Athletes Over Time </h3>
<p>Testing to see how the proportion of female athletes at the Olympics has changed since 1960, separately for Summer and Winter Games.
A statistically significant upward trend would suggest the Olympics have become meaningfully more gender inclusive over time.</p>

In [ ]:
import numpy as np

genderByYear = (biography_pruned.drop_duplicates(subset=['athlete_id', 'year', 'edition'])
    .groupby(['year', 'edition', 'sex'])['athlete_id']
    .nunique()
    .reset_index(name='count')
)


genderPivot = (
    genderByYear
    .pivot_table(index=['year', 'edition'], columns='sex', values='count', fill_value=0)
    .reset_index()
)
genderPivot.columns.name = None
genderPivot['total']      = genderPivot['Male'] + genderPivot['Female']
genderPivot['female_pct'] = genderPivot['Female'] / genderPivot['total'] * 100


genderPivot['season'] = genderPivot['edition'].apply(
    lambda x: 'Summer' if 'Summer' in str(x) else 'Winter'
)
summer = genderPivot[genderPivot['season'] == 'Summer'].sort_values('year')
winter = genderPivot[genderPivot['season'] == 'Winter'].sort_values('year')

# Linear regression, gives p values but its 0 so idk if its necessary to include them
sSlope, sInt, _, sP, _ = stats.linregress(summer['year'], summer['female_pct'])
wSlope, wInt, _, wP, _ = stats.linregress(winter['year'], winter['female_pct'])

print(f"Summer Games  — trend: {sSlope:+.2f}% female athletes per year  (p = {sP:.5f})")
print(f"Winter Games  — trend: {wSlope:+.2f}% female athletes per year  (p = {wP:.5f})")

# Plot 
fig, ax = plt.subplots(figsize=(13, 6))

ax.fill_between(summer['year'], summer['female_pct'], alpha=0.12, color='red')
ax.fill_between(winter['year'], winter['female_pct'], alpha=0.12, color='blue')

ax.plot(summer['year'], summer['female_pct'],
        color='red', linewidth=2.5, label='Summer Games')
ax.plot(winter['year'], winter['female_pct'],
        color='blue', linewidth=2.5, label='Winter Games')

yr_range_s = np.array([summer['year'].min(), summer['year'].max()])
yr_range_w = np.array([winter['year'].min(), winter['year'].max()])
ax.plot(yr_range_s, sSlope * yr_range_s + sInt,
        color='red', linestyle='--', linewidth=1.4, alpha=0.7)
ax.plot(yr_range_w, wSlope * yr_range_w + wInt,
        color='blue', linestyle='--', linewidth=1.4, alpha=0.7)


ax.set_xlim(summer['year'].min() - 2, summer['year'].max() + 4)
ax.set_ylim(0, 60)
ax.set_xlabel('Olympic Year', fontsize=13, fontweight='bold')
ax.set_ylabel('Female Athletes (%)', fontsize=13, fontweight='bold')
ax.set_title(
    'Female Athlete Participation at the Olympics Over Time\n'
    '(Dashed lines show linear trend)',
    fontsize=15, fontweight='bold'
)
ax.legend(fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


<h3> Conclusion #3 – Olympic medals won by athlete sex in mixed events </h3>

We tested whether athlete sex was a statistically significant influence to win medals in mixed-gender Olympic events.

In [ ]:

medals_by_gender_df = pd.DataFrame()

# Get mixed events
mixed_events = [event for event in events_df["event"].unique() if "Mixed" in event]
mixed_events_df = events_df[(events_df["event"].isin(mixed_events)) & (events_df["medal"] != "No Medal")].copy()

# Get athlete sex
def get_athlete_sex(id):
    return biography_df.loc[biography_df["athlete_id"] == id, "sex"].iloc[0]
mixed_events_df["sex"] = mixed_events_df["athlete_id"].map(get_athlete_sex)

# Get medal counts for mixed events by sex
medals_by_gender_df = mixed_events_df.groupby(["event", "sex"])["athlete_id"].nunique().unstack()
print("DATAFRAME")
print("Olympic medals won by sex in mixed events")
display(medals_by_gender_df)

# State hypotheses
print("HYPOTHESES")
print("H(0): Men and women win equal medals on average in mixed Olympic events.")
print("H(1): Men and women do not win equal medals on average in mixed Olympic events.")

# Run binomial test
medals_by_gender_count = mixed_events_df["sex"].value_counts()
male_count, female_count = medals_by_gender_count.get("Male", 0), medals_by_gender_count.get("Female", 0)
p_value = stats.binomtest(male_count, n=(male_count + female_count), p=0.5, alternative="two-sided")
print("\nBINOMIAL TEST")
print("Expected proportion:", 0.5)
print("Actual proportion:", p_value.statistic)
print("p-value:", p_value.pvalue)
print("\nBecause the p-value is greater than 0.05, we do not reject H(0). There is no gender imbalance in medal-winning athletes for mixed Olympic sports.")

# Plot data
print("\nPLOT")

medals_by_gender_df["Total"] = medals_by_gender_df["Male"] + medals_by_gender_df["Female"]
medals_by_gender_df = medals_by_gender_df.sort_values("Total", ascending=True)

mixed_events = list(medals_by_gender_df.index)
male_count, female_count = list(medals_by_gender_df["Male"]), list(medals_by_gender_df["Female"])

plt.figure(figsize=(12,6))

plt.bar(mixed_events, male_count, color='#bde0fe', label='Men')
plt.bar(mixed_events, female_count, bottom=male_count, color='#ffc8dd', label='Women')

plt.ylabel("Number of medals", fontweight='bold')
plt.xlabel("Event", fontweight='bold')
plt.title("Olympic medals won by sex in mixed events", fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right', size=7)
plt.legend(title="Athlete sex", reverse=True)


plt.tight_layout()
plt.show()

# 5. Primary Analysis

## 5.1 Machine Learning Design & Development

<h3><i>Goal: Figure out whether an athlete can win a medal in their event</i></h3>

We chose classification as our machine learning technique since we want to find out whether an athlete falls into 1 of 2 categories (if they medal or not). Our EDA showed that factors like sex and country of origin can affect whether an athlete medals or not so we decided to use models that can be used for classification like Logistic Regression and tree based models like Decision Trees, Extra Trees, Gradient Boosting and AdaBoost, since they capture more complex relationships than just average regression and can tell us what features are the most useful for prediction.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score,
)

# build the modeling dataframe from biography_pruned and encode the target.
# we treat any medal (gold, silver, bronze) as a positive outcome.
ml_df = biography_pruned.copy()

# 1 = any medal, 0 = no medal
ml_df["medal_won"] = (ml_df["medal"] != "No Medal").astype(int)

ml_df["sex"]         = ml_df["sex"].map({"Male": 0, "Female": 1})
ml_df["isTeamSport"] = ml_df["isTeamSport"].astype(int)

# engineer three career-trajectory features. career trajectory ends up more
# predictive than raw physical attributes like height and weight.
ml_df["age"] = ml_df["year"] - ml_df["born"].dt.year  # athletes peak at certain ages

ml_df = ml_df.sort_values(["athlete_id", "year"]).reset_index(drop=True)

ml_df["prior_appearances"] = ml_df.groupby("athlete_id").cumcount()  # experience proxy

# subtract the current row so we only see what happened in prior games
ml_df["prior_medals"] = (
    ml_df.groupby("athlete_id")["medal_won"].cumsum() - ml_df["medal_won"]
)

# add two event-level features that describe the structure of the event itself,
# independent of the individual athlete.
# fraction of all-time entries in this event that medaled
event_medal_rate = (
    biography_pruned.groupby("event")["medal"]
    .apply(lambda x: (x != "No Medal").mean())
    .rename("event_medal_rate")
)
# average field size per edition; a bigger field means harder to podium
event_avg_size = (
    biography_pruned.groupby(["event", "year"])["athlete_id"]
    .nunique()
    .groupby("event").mean()
    .rename("event_avg_size")
)
ml_df = ml_df.join(event_medal_rate, on="event")
ml_df = ml_df.join(event_avg_size,   on="event")

numeric_features     = ["height", "weight", "year", "sex", "isTeamSport",
                        "age", "prior_appearances", "prior_medals",
                        "event_medal_rate", "event_avg_size"]
categorical_features = ["country", "sport"]

X = ml_df[numeric_features + categorical_features].copy()
Y = ml_df["medal_won"].copy()

mask = X.notna().all(axis=1) & Y.notna()
X, Y = X[mask], Y[mask]

print(f"Rows available for ML: {len(X):,}")
print(f"Medal rate: {Y.mean():.3f}")
print(Y.value_counts(normalize=True).rename({0: "No Medal", 1: "Medal"}).round(3).to_string())

# define five classifiers spanning linear to gradient-boosted.
# if they all plateau at the same f1, the bottleneck is the features, not the algorithm.
# class_weight="balanced" up-weights medals so models don't just predict "no medal" for everything
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, class_weight="balanced", random_state=42
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100, max_depth=12, class_weight="balanced",
        random_state=42, n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=12, class_weight="balanced",
        random_state=42, n_jobs=-1
    ),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.05, class_weight="balanced",
        random_state=42
    ),
}


## 5.2 Algorithm Training & Test Data Analysis

In [ ]:
# quick sanity check before the expensive cv step.
# verify the shape of the feature matrix and confirm the class distribution looks right.
print(f"Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns")
print(f"Medals (positive class): {Y.sum():,} / {len(Y):,} ({Y.mean():.1%})\n")
for col in X.columns:
    print(f"  {col:<22}  dtype={str(X[col].dtype):<10}  nulls={X[col].isna().sum()}")

# confirm no NaNs remain before passing data to sklearn.
# any NaNs here will silently break model fitting.
nan_counts = X.isna().sum()
if nan_counts.sum() != 0:
    print(f"Unexpected NaNs:\n{nan_counts[nan_counts > 0]}")



Ensuring no NaNs that can affect model fitting in the following step

In [ ]:
# confirm no NaNs remain before passing data to sklearn.
# any NaNs here will silently break model fitting.
nan_counts = X.isna().sum()
assert nan_counts.sum() == 0, f"Unexpected NaNs:\n{nan_counts[nan_counts > 0]}"


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate

# split into train and test sets, then run 5-fold cross-validation on the training set.
# we pick the best model by cv f1 and refit it on all of X_train before evaluating on X_test.

# stratify keeps the 86/14 imbalance consistent across both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

# TargetEncoder instead of OHE: country has 200+ unique values so OHE would create
# a massive sparse matrix. maps each category to its historical medal rate instead,
# and is fit on training folds only to avoid leakage.
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", TargetEncoder(target_type="binary", random_state=42), categorical_features),
])

# f1 not accuracy: 86% of entries are "no medal" so accuracy would be misleadingly high
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"accuracy": "accuracy", "precision": "precision",
           "recall": "recall", "f1": "f1"}

results        = []
trained_models = {}

for name, model in models.items():
    print(f"  Training {name}...")
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier",   model),
    ])
    cv_scores = cross_validate(
        pipe, X_train, y_train,
        cv=kfold, scoring=scoring,
        return_train_score=False, n_jobs=-1,
    )
    results.append({
        "Model":     name,
        "Accuracy":  cv_scores["test_accuracy"].mean(),
        "Precision": cv_scores["test_precision"].mean(),
        "Recall":    cv_scores["test_recall"].mean(),
        "F1":        cv_scores["test_f1"].mean(),
    })
    pipe.fit(X_train, y_train)
    trained_models[name] = pipe
    print(f"    CV F1: {results[-1]['F1']:.4f}")

results_df = (
    pd.DataFrame(results)
    .set_index("Model")
    .sort_values("F1", ascending=False)
)
print("\nCross-Validation Results (5-fold)")
display(results_df.round(4))


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# evaluate every model on the held-out test set and pick the best by cv f1.
# reporting all models, not just the winner, shows whether added complexity is buying anything.
best_name  = results_df["F1"].idxmax()
best_model = trained_models[best_name]
print(f"Best model by CV F1: {best_name}\n")

test_rows = []
for name, model in trained_models.items():
    yp = model.predict(X_test)
    test_rows.append({
        "Model":     name,
        "Accuracy":  accuracy_score(y_test, yp),
        "Precision": precision_score(y_test, yp),
        "Recall":    recall_score(y_test, yp),
        "F1":        f1_score(y_test, yp),
    })

test_df = (
    pd.DataFrame(test_rows)
    .set_index("Model")
    .sort_values("F1", ascending=False)
)
print("Test Set Performance — All Models")
display(test_df.round(4))

y_pred_best = best_model.predict(X_test)
print(f"\nClassification Report: {best_name}")
print(classification_report(y_test, y_pred_best, target_names=["No Medal", "Medal"]))

# rows = actual, columns = predicted; top-right = false alarms, bottom-left = missed medalists
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=["No Medal", "Medal"],
    cmap="Blues", ax=ax,
)
ax.set_title(f"Confusion Matrix — {best_name}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


Confusion matrix here confirms what we've been saying: there are clearly more "no medal" classifications given out. This is true as there are generally less people that receive medals, but it leaves room for class imbalances and biases when training.

In [ ]:
from sklearn.metrics import precision_recall_curve

# the default 0.5 decision threshold is rarely optimal when classes are imbalanced.
# we use the precision-recall curve to find the threshold that maximizes f1 on the test set,
# then apply it to get a final tuned prediction.

# 0.5 isn't optimal with class imbalance, so we scan all thresholds
# and pick the one that maximizes f1 directly
y_proba = best_model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

# n+1 values returned for n thresholds, so drop the last element before argmax
f1s = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1s[:-1])
best_threshold = thresholds[best_idx]

print(f"default threshold (0.5) F1:  {f1_score(y_test, best_model.predict(X_test)):.4f}")
print(f"optimal threshold:           {best_threshold:.3f}")
print(f"optimal threshold F1:        {f1s[best_idx]:.4f}")

y_pred_tuned = (y_proba >= best_threshold).astype(int)
print(f"\n{classification_report(y_test, y_pred_tuned, target_names=['No Medal', 'Medal'])}")

# moving left on the curve catches more medalists but also raises false alarms
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recalls[:-1], precisions[:-1], lw=2, label="precision-recall curve")
ax.scatter(
    recalls[best_idx], precisions[best_idx],
    color="red", s=100, zorder=5,
    label=f"best threshold = {best_threshold:.3f}  (F1 = {f1s[best_idx]:.3f})",
)
ax.set_xlabel("Recall", fontweight="bold")
ax.set_ylabel("Precision", fontweight="bold")
ax.set_title(f"Precision-Recall Curve — {best_name}", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


<h2>6. Visualization and Result Analysis</h2>

<p>
    We produce four visualizations that together explain both how well the model
    predicts and what it learned:

Confusion Matrix: counts predictions against ground truth on the
held-out test set. The off-diagonal cells show the cost of each type of mistake:
false positives (non-medalists the model flagged) vs. false negatives (medalists
the model missed).

Precision-Recall Curve: shows the tradeoff between precision and
recall at every possible decision threshold. Moving the threshold down catches more
medalists (higher recall) but also increases false alarms (lower precision). The
red dot marks the threshold that maximizes F1.

Feature Importances: ranks each input by its average decrease in
node impurity across all trees. Engineered features are highlighted in orange to
show their incremental contribution over the original biographic attributes.

ROC Curves: plots True Positive Rate vs. False Positive Rate for
all five classifiers on the same axes. AUC summarizes discriminative ability
independent of threshold: AUC = 0.5 is random chance, AUC = 1.0 is perfect.`

</p>
 


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# two plots side by side: feature importances from the best model on the left,
# and roc curves for all five classifiers on the right.
# together they show what the model learned and how well each classifier separates the classes.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# single column per category so importances are readable without aggregating OHE dummies
classifier = best_model.named_steps["classifier"]
all_feat_names = np.array(numeric_features + categorical_features)

if hasattr(classifier, "feature_importances_"):
    importances = classifier.feature_importances_
elif hasattr(classifier, "coef_"):
    importances = np.abs(classifier.coef_[0])
else:
    importances = np.zeros(len(all_feat_names))

engineered = {"age", "prior_appearances", "prior_medals"}
sorted_idx = np.argsort(importances)
bar_colors = [
    "#e76f51" if all_feat_names[i] in engineered else "#264653"
    for i in sorted_idx
]
axes[0].barh(all_feat_names[sorted_idx], importances[sorted_idx], color=bar_colors)
axes[0].legend(
    handles=[
        Patch(facecolor="#e76f51", label="Engineered feature"),
        Patch(facecolor="#264653", label="Original feature"),
    ],
    fontsize=9,
)
axes[0].set_xlabel("Feature Importance (mean decrease in impurity)", fontweight="bold")
axes[0].set_title(f"Feature Importances — {best_name}", fontsize=13, fontweight="bold")

# auc is threshold-free so it's a cleaner cross-model comparison than f1
for name, model in trained_models.items():
    clf = model.named_steps["classifier"]
    if hasattr(clf, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(clf, "decision_function"):
        y_score = model.decision_function(X_test)
    else:
        continue
    fpr, tpr, _ = roc_curve(y_test, y_score)
    auc = roc_auc_score(y_test, y_score)
    axes[1].plot(fpr, tpr, lw=2, linestyle="--" if name == best_name else "-",
                 label=f"{name}  (AUC = {auc:.3f})")

axes[1].plot([0, 1], [0, 1], "k:", lw=1, label="Random classifier (AUC = 0.500)")
axes[1].set_xlabel("False Positive Rate", fontsize=12, fontweight="bold")
axes[1].set_ylabel("True Positive Rate", fontsize=12, fontweight="bold")
axes[1].set_title("ROC Curves — All Models", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=8, loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


<h2> Conclusion</h2>
<p>

This project analyzed 126 years of Olympic data to predict whether an athlete would win a medal based on their physical profile, country of origin, sport, and event. The analysis covered three stages: exploratory data analysis with hypothesis testing, feature engineering, and a multi-model classification pipeline.

We tested three hypotheses in our EDA and Hypothesis Testing. A paired t-test confirmed that host nations win statistically significantly more medals at home than away. Linear regression on female athlete participation showed a significant upward trend in both Summer and Winter Games since 1960. A binomial test on mixed gender events found no significant difference in medals won by men versus women. Each of these findings suggested that country, sex, and event structure carry some signal in medal outcomes, which motivated their inclusion as features in the model. We dropped records before 1960 because height and weight missingness exceeded 45% for those years. Imputing that volume of data would have introduced bias. Remaining missing height and weight values were filled using per-sex, per-sport means. Weight entries stored as ranges were converted to midpoint averages. Country codes were merged in from a separate lookup table to produce a single flat modeling dataset. We added three additional features: athlete age, prior Olympic appearances, and prior medals won. Two event-level features were also computed: the historical medal rate for each event and the average field size per edition. These additions gave the models more signal than raw physical attributes alone.

Four classifiers were trained and evaluated under 5-fold stratified cross-validation: Logistic Regression, Decision Tree, Extra Trees, and Random Forest. F1 score was used as the primary metric given the class imbalance that approximately 86.5% of entries are no-medal. The best model achieved an F1 of approximately 0.39 in the medal class. All models performed similarly, with no single classifier substantially outperforming the others. The class imbalance (roughly 13.5% positive class) makes this a difficult classification problem by construction. Team sport entries complicate the target variable, since multiple athletes on a relay team all register as medal outcomes despite the result being a single team performance. Performance data beyond final placement is also absent from the dataset.
The low F1 across all models is the main result of our analysis. Despite including physical attributes, demographic information, career history, and event-level features, no classifier was able to predict medal outcomes with strong accuracy. This suggests that the features available in this dataset, height, weight, age, country, sport, and prior appearances, are fundamentally weak predictors of whether an athlete will medal. Olympic medal outcomes appear to be driven largely by factors this dataset does not capture, such as training quality, in-competition performance, and event-specific competitive depth. The inability to predict medals well is itself a meaningful finding: it indicates that no simple demographic or physical profile reliably separates medalists from non-medalists.

Summary: Across all models tested, medal prediction from physical and demographic features alone plateaus around an F1 of 0.39. The EDA results show that country and event structure carry some signal, but not enough to produce reliable individual-level predictions. The main takeaway is that Olympic medal outcomes are not well predicted by the attributes measured here, which points to the importance of performance-based and training-based data that this dataset does not include.
</p>